# 03 — XPath et XSLT

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- naviguer dans un XML avec des expressions XPath ;
- utiliser les prédicats, axes et fonctions XPath ;
- gérer les namespaces dans les requêtes XPath ;
- transformer un XML en HTML/texte avec XSLT ;
- passer des paramètres aux feuilles XSLT ;
- utiliser `apply-templates` et les modes.

## Prérequis — ce que vous devez maîtriser

À ce stade de la formation, vous savez :

- bases XML (parsing avec `lxml`) ;
- lxml installé.

Ce que ce notebook **ne couvrira pas** :

- XQuery ;
- XProc (pipelines XML) ;
- XPath 2.0+ et 3.0+ (nécessitent Saxon).

## Plan

1. Installation
2. XPath de base
3. Prédicats
4. Axes XPath
5. Fonctions XPath
6. XPath avec namespaces
7. XSLT — premier template
8. Variables et paramètres XSLT
9. Apply-templates
10. Conditions et boucles XSLT
11. Synthèse
12. Exercices
13. Ressources externes

### Exemple avancé : XSLT avec tri et filtrage

XSLT permet de trier et filtrer les résultats directement
dans la transformation, sans code Python.

In [ ]:
from lxml import etree

xml_src = etree.fromstring(b'''
<bibliotheque>
  <livre genre="SF" annee="1965"><titre>Dune</titre><auteur>Herbert</auteur></livre>
  <livre genre="Roman" annee="1862"><titre>Les Misérables</titre><auteur>Hugo</auteur></livre>
  <livre genre="SF" annee="1951"><titre>Fondation</titre><auteur>Asimov</auteur></livre>
  <livre genre="Roman" annee="1844"><titre>Les Trois Mousquetaires</titre><auteur>Dumas</auteur></livre>
  <livre genre="SF" annee="1968"><titre>2001</titre><auteur>Clarke</auteur></livre>
</bibliotheque>
''')

xslt_src = etree.fromstring(b'''
<xsl:stylesheet version="1.0" xmlns:xsl="http://www.w3.org/1999/XSL/Transform">
  <xsl:template match="/bibliotheque">
    <html><body>
      <h1>Livres de SF (triés par année)</h1>
      <ul>
        <xsl:for-each select="livre[@genre='SF']">
          <xsl:sort select="@annee"/>
          <li><xsl:value-of select="titre"/> (<xsl:value-of select="@annee"/>) — <xsl:value-of select="auteur"/></li>
        </xsl:for-each>
      </ul>
    </body></html>
  </xsl:template>
</xsl:stylesheet>
''')

transform = etree.XSLT(xslt_src)
result = transform(xml_src)
print(etree.tostring(result, pretty_print=True, encoding='unicode'))

### Exercice : XPath complexe

Sur le document `bibliotheque` ci-dessus, écrivez les requêtes XPath pour :

1. Tous les titres de livres publiés après 1900
2. Les auteurs de genre 'Roman' triés alphabétiquement
3. Le nombre total de livres par genre
4. Le livre le plus ancien

In [ ]:
# 1. Titres après 1900
titres_recents = xml_src.xpath('//livre[@annee > 1900]/titre/text()')
print('Après 1900 :', titres_recents)

# 2. Auteurs de romans
auteurs_roman = xml_src.xpath('//livre[@genre="Roman"]/auteur/text()')
print('Auteurs Roman :', sorted(auteurs_roman))

# 3. Nombre par genre
for genre in ['SF', 'Roman']:
    count = xml_src.xpath(f'count(//livre[@genre="{genre}"])')
    print(f'{genre} : {int(count)} livres')

# 4. Le plus ancien
oldest = xml_src.xpath('//livre[not(../livre/@annee < @annee)]/titre/text()')
print('Plus ancien :', oldest)

### Exercice : transformation XSLT vers Markdown

Écrivez une feuille XSLT qui transforme le catalogue de livres en format Markdown :
```markdown
# Catalogue

## SF
- **Dune** (1965) par Herbert
...
```

Hint : utilisez `xsl:output method="text"` et `xsl:text` pour le formatage.

In [ ]:
xslt_md = etree.fromstring(b'''
<xsl:stylesheet version="1.0" xmlns:xsl="http://www.w3.org/1999/XSL/Transform">
  <xsl:output method="text"/>
  <xsl:template match="/bibliotheque">
    <xsl:text># Catalogue\n\n</xsl:text>
    <xsl:for-each select="livre[not(@genre = preceding-sibling::livre/@genre)]">
      <xsl:variable name="g" select="@genre"/>
      <xsl:text>## </xsl:text><xsl:value-of select="$g"/><xsl:text>\n</xsl:text>
      <xsl:for-each select="/bibliotheque/livre[@genre=$g]">
        <xsl:sort select="@annee"/>
        <xsl:text>- **</xsl:text><xsl:value-of select="titre"/>
        <xsl:text>** (</xsl:text><xsl:value-of select="@annee"/>
        <xsl:text>) par </xsl:text><xsl:value-of select="auteur"/><xsl:text>\n</xsl:text>
      </xsl:for-each>
      <xsl:text>\n</xsl:text>
    </xsl:for-each>
  </xsl:template>
</xsl:stylesheet>
''')

transform_md = etree.XSLT(xslt_md)
result_md = transform_md(xml_src)
print(str(result_md))

---

## 1. Installation

In [ ]:
from lxml import etree
import lxml
print(f"lxml {lxml.__version__}")

---

## 2. XPath de base

In [ ]:
xml = '''<catalog>
  <book id='1'>
    <title>Python</title>
    <price>29.90</price>
    <author>Alice</author>
  </book>
  <book id='2'>
    <title>Rust</title>
    <price>34.50</price>
    <author>Bob</author>
  </book>
  <book id='3'>
    <title>Go</title>
    <price>24.00</price>
    <author>Alice</author>
  </book>
</catalog>'''
doc = etree.fromstring(xml)

In [ ]:
# Chemin absolu
doc.xpath('/catalog/book/title/text()')

In [ ]:
# Recherche globale
doc.xpath('//title/text()')

In [ ]:
# Attributs
doc.xpath('/catalog/book/@id')

In [ ]:
# Tous les auteurs (sans doublon via set)
list(set(doc.xpath('//author/text()')))

---

## 3. Prédicats

Filtrer les résultats avec `[condition]`.

In [ ]:
# Par attribut
doc.xpath("//book[@id='1']/title/text()")

In [ ]:
# Par valeur numérique
doc.xpath('//book[price > 30]/title/text()')

In [ ]:
# Par position (1-indexed !)
doc.xpath('//book[1]/title/text()')

In [ ]:
# Dernier élément
doc.xpath('//book[last()]/title/text()')

In [ ]:
# Par contenu textuel
doc.xpath("//book[author='Alice']/title/text()")

---

## 4. Axes XPath

Les axes permettent de naviguer dans différentes directions.

| Axe | Raccourci | Signification |
|---|---|---|
| `child::` | *(défaut)* | Enfants directs |
| `parent::` | `..` | Parent |
| `ancestor::` | | Tous les ancêtres |
| `descendant::` | `.//` | Tous les descendants |
| `following-sibling::` | | Frères suivants |
| `preceding-sibling::` | | Frères précédents |
| `attribute::` | `@` | Attributs |
| `self::` | `.` | L’élément lui-même |

In [ ]:
# Parent du premier title
first_title = doc.xpath('//title')[0]
first_title.getparent().get('id')

In [ ]:
# Frères suivants du premier book
first_book = doc.xpath('//book[1]')[0]
[b.xpath('title/text()')[0] for b in first_book.itersiblings()]

---

## 5. Fonctions XPath

In [ ]:
# count
doc.xpath('count(//book)')

In [ ]:
# sum
doc.xpath('sum(//price)')

In [ ]:
# contains
doc.xpath("//book[contains(title, 'Py')]/title/text()")

In [ ]:
# string-length
doc.xpath('string-length(//title[1])')

In [ ]:
# concat
doc.xpath("concat(//book[1]/title, ' by ', //book[1]/author)")

In [ ]:
# normalize-space
doc.xpath("normalize-space('  hello   world  ')")

---

## 6. XPath avec namespaces

In [ ]:
ns_xml = '<root xmlns="http://example.com"><item>Hello</item><item>World</item></root>'
ns_doc = etree.fromstring(ns_xml)

# Sans namespace : ne trouve rien
print('Sans NS:', ns_doc.xpath('//item'))

In [ ]:
# Avec namespace
ns = {'ex': 'http://example.com'}
print('Avec NS:', ns_doc.xpath('//ex:item/text()', namespaces=ns))

---

## 7. XSLT — premier template

XSLT transforme un XML en un autre format (HTML, texte, XML).

In [ ]:
xslt_src = '''<?xml version='1.0'?>
<xsl:stylesheet version='1.0' xmlns:xsl='http://www.w3.org/1999/XSL/Transform'>
  <xsl:template match='/'>
    <html>
      <body>
        <h1>Catalogue</h1>
        <ul>
          <xsl:for-each select='catalog/book'>
            <li><xsl:value-of select='title'/> - <xsl:value-of select='price'/> EUR</li>
          </xsl:for-each>
        </ul>
      </body>
    </html>
  </xsl:template>
</xsl:stylesheet>'''

transform = etree.XSLT(etree.fromstring(xslt_src))
result = transform(doc)
print(str(result)[:300])

---

## 8. Variables et paramètres XSLT

In [ ]:
xslt_param = '''<?xml version='1.0'?>
<xsl:stylesheet version='1.0' xmlns:xsl='http://www.w3.org/1999/XSL/Transform'>
  <xsl:param name='currency'>EUR</xsl:param>
  <xsl:template match='/'>
    <total><xsl:value-of select="sum(catalog/book/price)"/> <xsl:value-of select='$currency'/></total>
  </xsl:template>
</xsl:stylesheet>'''

transform = etree.XSLT(etree.fromstring(xslt_param))
result = transform(doc, currency="'USD'")
print(str(result))

In [ ]:
# Paramètre par défaut (EUR)
result_default = transform(doc)
print(str(result_default))

---

## 9. Apply-templates

Plus puissant que `for-each` : permet la récursion et les modes.

In [ ]:
xslt_apply = '''<?xml version='1.0'?>
<xsl:stylesheet version='1.0' xmlns:xsl='http://www.w3.org/1999/XSL/Transform'>
  <xsl:template match='/'>
    <result>
      <xsl:apply-templates select='catalog/book'/>
    </result>
  </xsl:template>
  <xsl:template match='book'>
    <entry id='{@id}'>
      <xsl:value-of select='title'/> by <xsl:value-of select='author'/>
    </entry>
  </xsl:template>
</xsl:stylesheet>'''

transform = etree.XSLT(etree.fromstring(xslt_apply))
print(str(transform(doc)))

---

## 10. Conditions et boucles XSLT

In [ ]:
xslt_cond = '''<?xml version='1.0'?>
<xsl:stylesheet version='1.0' xmlns:xsl='http://www.w3.org/1999/XSL/Transform'>
  <xsl:template match='/'>
    <cheap>
      <xsl:for-each select='catalog/book'>
        <xsl:sort select='price' data-type='number'/>
        <xsl:if test='price &lt; 30'>
          <book><xsl:value-of select='title'/></book>
        </xsl:if>
      </xsl:for-each>
    </cheap>
  </xsl:template>
</xsl:stylesheet>'''

transform = etree.XSLT(etree.fromstring(xslt_cond))
print(str(transform(doc)))

---

## Synthèse

| Opération | lxml |
|---|---|
| XPath | `doc.xpath('expression')` |
| Namespaces | `doc.xpath('//ex:tag', namespaces={'ex': 'uri'})` |
| XSLT compile | `transform = etree.XSLT(stylesheet)` |
| XSLT applique | `result = transform(doc, param='value')` |

### XPath essentiel

| Syntaxe | Signification |
|---|---|
| `/a/b` | Chemin absolu |
| `//b` | N’importe où |
| `@attr` | Attribut |
| `[1]` | Premier (1-indexed) |
| `[last()]` | Dernier |
| `[@id='x']` | Prédicat sur attribut |
| `[price > 30]` | Prédicat numérique |
| `contains(x, 'y')` | Fonction |
| `text()` | Contenu textuel |
| `count()`, `sum()` | Agrégation |

### Règles à retenir

- **XPath 1.0** est le standard lxml (pour 2.0+/3.0+, utiliser Saxon).
- **Namespaces** doivent être déclarés explicitement dans le dict.
- **XSLT** : on compile d’abord, puis on applique.
- **`apply-templates`** > `for-each` pour la modularité.

---

---

## XPath : position et regroupement

In [ ]:
from lxml import etree
xml = '''<catalog>
  <book id='1'><title>Python</title><price>29.90</price><author>Alice</author></book>
  <book id='2'><title>Rust</title><price>34.50</price><author>Bob</author></book>
  <book id='3'><title>Go</title><price>24.00</price><author>Alice</author></book>
</catalog>'''
doc = etree.fromstring(xml)

# Position
print('Premier:', doc.xpath('//book[position()=1]/title/text()'))
print('Deux derniers:', doc.xpath('//book[position()>=last()-1]/title/text()'))

In [ ]:
# Compter par auteur
for author in set(doc.xpath('//author/text()')):
    count = doc.xpath(f"count(//book[author='{author}'])")
    print(f"{author}: {int(count)} livres")

---

## XSLT : modes

Les modes XSLT permettent d'appliquer differents templates au meme element selon le contexte.

In [ ]:
xslt_mode = '''<?xml version='1.0'?>
<xsl:stylesheet version='1.0' xmlns:xsl='http://www.w3.org/1999/XSL/Transform'>
  <xsl:template match='/'>
    <result>
      <titles><xsl:apply-templates select='catalog/book' mode='title-only'/></titles>
      <details><xsl:apply-templates select='catalog/book' mode='full'/></details>
    </result>
  </xsl:template>
  <xsl:template match='book' mode='title-only'>
    <t><xsl:value-of select='title'/></t>
  </xsl:template>
  <xsl:template match='book' mode='full'>
    <entry><xsl:value-of select='title'/> (<xsl:value-of select='price'/> EUR)</entry>
  </xsl:template>
</xsl:stylesheet>'''
transform = etree.XSLT(etree.fromstring(xslt_mode))
print(str(transform(doc))[:300])

---

## XPath : operateurs booleens

In [ ]:
# AND / OR
print(doc.xpath("//book[price > 25 and author='Alice']/title/text()"))
print(doc.xpath("//book[price < 25 or price > 33]/title/text()"))

In [ ]:
# NOT
print(doc.xpath("//book[not(author='Alice')]/title/text()"))

---

## Extensions XPath avec lxml

In [ ]:
# Fonction XPath personnalisee
def upper(context, nodes):
    return [n.upper() for n in nodes]

ns = etree.FunctionNamespace(None)
ns['upper'] = upper
result = doc.xpath('upper(//title/text())')
print(result)

## Exercices

### Exercice 1 *(facile)* — Lister les titres

Extrayez tous les `title` du catalogue XML défini plus haut.

In [ ]:
from lxml import etree

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
from lxml import etree

xml = '<c><b><title>A</title></b><b><title>B</title></b><b><title>C</title></b></c>'
doc = etree.fromstring(xml)
print(doc.xpath('//title/text()'))
```

</details>

### Exercice 2 *(facile)* — Filtrer par attribut

Filtrez les `item` ayant `status='active'`.

In [ ]:
from lxml import etree

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
from lxml import etree

xml = "<r><i status='active'>A</i><i status='pending'>B</i><i status='active'>C</i></r>"
doc = etree.fromstring(xml)
print([e.text for e in doc.xpath("//i[@status='active']")])
```

</details>

### Exercice 3 *(moyen)* — Somme XPath

Avec XPath, calculez la somme des `price` dans le catalogue.

In [ ]:
from lxml import etree

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
from lxml import etree

xml = '<c><b><price>10</price></b><b><price>20</price></b><b><price>30</price></b></c>'
doc = etree.fromstring(xml)
print(f"Somme: {doc.xpath('sum(//price)')}")
```

</details>

### Exercice 4 *(moyen)* — XSLT vers HTML

Transformez un XML de personnes en tableau HTML `<table><tr><td>nom</td><td>age</td></tr>...</table>`.

In [ ]:
from lxml import etree

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
from lxml import etree

xml = '<personnes><p nom="Alice" age="30"/><p nom="Bob" age="25"/></personnes>'
xslt = '''<xsl:stylesheet version='1.0' xmlns:xsl='http://www.w3.org/1999/XSL/Transform'>
<xsl:template match='/'><table>
<xsl:for-each select='personnes/p'>
<tr><td><xsl:value-of select='@nom'/></td><td><xsl:value-of select='@age'/></td></tr>
</xsl:for-each>
</table></xsl:template>
</xsl:stylesheet>'''
transform = etree.XSLT(etree.fromstring(xslt))
print(str(transform(etree.fromstring(xml))))
```

</details>

### Exercice 5 *(difficile)* — XSLT avec paramètre et tri

Transformez le catalogue en HTML avec un paramètre `min_price` qui filtre les livres trop bon marché, et triez par prix décroissant.

In [ ]:
from lxml import etree

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
from lxml import etree

xml = '<c><b><title>A</title><price>10</price></b><b><title>B</title><price>30</price></b><b><title>C</title><price>20</price></b></c>'
xslt = '''<xsl:stylesheet version='1.0' xmlns:xsl='http://www.w3.org/1999/XSL/Transform'>
<xsl:param name='min_price'>15</xsl:param>
<xsl:template match='/'><ul>
<xsl:for-each select='c/b[price >= $min_price]'>
<xsl:sort select='price' data-type='number' order='descending'/>
<li><xsl:value-of select='title'/>: <xsl:value-of select='price'/></li>
</xsl:for-each>
</ul></xsl:template>
</xsl:stylesheet>'''
transform = etree.XSLT(etree.fromstring(xslt))
print(str(transform(etree.fromstring(xml))))
```

</details>

### Approfondir : fonctions XPath d'extension avec lxml

lxml permet de définir des fonctions XPath personnalisées en Python,
ce qui étend considérablement les capacités de requêtage.

In [ ]:
from lxml import etree

# Fonction XPath personnalisée
def xpath_upper(context, nodes):
    """Convertit le texte en majuscules."""
    return [n.upper() if isinstance(n, str) else n for n in nodes]

ns = etree.FunctionNamespace(None)
ns['upper'] = xpath_upper

xml = etree.fromstring(b'<noms><nom>alice</nom><nom>bob</nom><nom>charlie</nom></noms>')
result = xml.xpath('upper(//nom/text())')
print('Résultat :', result)

### Exercice : XSLT paramétré

XSLT permet de passer des paramètres depuis Python.
Créez une transformation qui filtre les livres par genre,
le genre étant passé en paramètre.

In [ ]:
from lxml import etree

xml_bib = etree.fromstring(b'''
<bibliotheque>
  <livre genre="SF"><titre>Dune</titre></livre>
  <livre genre="Roman"><titre>1984</titre></livre>
  <livre genre="SF"><titre>Fondation</titre></livre>
  <livre genre="Polar"><titre>Millenium</titre></livre>
</bibliotheque>
''')

xslt = etree.fromstring(b'''
<xsl:stylesheet version="1.0" xmlns:xsl="http://www.w3.org/1999/XSL/Transform">
  <xsl:param name="genre_filtre"/>
  <xsl:template match="/bibliotheque">
    <resultats>
      <xsl:for-each select="livre[@genre=$genre_filtre]">
        <titre><xsl:value-of select="titre"/></titre>
      </xsl:for-each>
    </resultats>
  </xsl:template>
</xsl:stylesheet>
''')

transform = etree.XSLT(xslt)
for genre in ['SF', 'Roman', 'Polar']:
    result = transform(xml_bib, genre_filtre=etree.XSLT.strparam(genre))
    titres = result.xpath('//titre/text()')
    print(f'{genre}: {titres}')

### Exercice récapitulatif : pipeline complet

1. Parsez un XML de produits
2. Utilisez XPath pour extraire les produits dont le prix > 100
3. Appliquez une XSLT pour générer un rapport HTML
4. Affichez le HTML résultant

In [ ]:
from lxml import etree

xml_produits = etree.fromstring(b'''
<produits>
  <produit><nom>Laptop</nom><prix>899</prix></produit>
  <produit><nom>Souris</nom><prix>29</prix></produit>
  <produit><nom>Ecran</nom><prix>349</prix></produit>
  <produit><nom>Cable</nom><prix>12</prix></produit>
  <produit><nom>Clavier</nom><prix>79</prix></produit>
  <produit><nom>Casque</nom><prix>199</prix></produit>
</produits>
''')

# XPath : produits chers
chers = xml_produits.xpath('//produit[prix > 100]')
print(f'{len(chers)} produits > 100 EUR')
for p in chers:
    print(f'  {p.findtext("nom")} : {p.findtext("prix")} EUR')

### Exercice bonus : XSLT pour rapport d'inventaire

Créez une XSLT qui transforme un catalogue produits en rapport HTML
avec un tableau triable, le montant total du stock et un avertissement
pour les produits en rupture.

In [ ]:
from lxml import etree

xml_inv = etree.fromstring(b'''
<inventaire>
  <produit><nom>Laptop</nom><prix>899</prix><stock>5</stock></produit>
  <produit><nom>Souris</nom><prix>29</prix><stock>0</stock></produit>
  <produit><nom>Ecran</nom><prix>349</prix><stock>12</stock></produit>
  <produit><nom>Cable</nom><prix>12</prix><stock>200</stock></produit>
  <produit><nom>Clavier</nom><prix>79</prix><stock>0</stock></produit>
</inventaire>
''')

# Requêtes XPath utiles
total_stock = xml_inv.xpath('sum(//produit/stock)')
ruptures = xml_inv.xpath('//produit[stock=0]/nom/text()')
valeur_totale = sum(
    float(p.findtext('prix')) * float(p.findtext('stock'))
    for p in xml_inv.xpath('//produit')
)

print(f'Stock total : {int(total_stock)} unités')
print(f'Valeur totale : {valeur_totale:.2f} EUR')
print(f'En rupture : {ruptures}')

### Astuce : déboguer les expressions XPath

- **`string()`** pour convertir un noeud en texte : `string(//livre[1]/titre)`
- **`count()`** pour compter : `count(//livre[@genre='SF'])`
- **`position()`** dans un prédicat : `//livre[position() <= 3]`
- **`last()`** : `//livre[last()]`
- **`contains()`** : `//livre[contains(titre, 'Python')]`
- **`starts-with()`** : `//livre[starts-with(auteur, 'H')]`

---

## Ressources externes

- [XPath 1.0 — W3C Specification](https://www.w3.org/TR/xpath-10/)
- [XSLT 1.0 — W3C Specification](https://www.w3.org/TR/xslt-10/)
- [lxml XPath tutorial](https://lxml.de/xpathxslt.html)
- [XPath cheat sheet](https://devhints.io/xpath)